# Numerical checks for Appendix E

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$This notebook numerically checks the constructions of the Szegedy quantum walk given in in Appendix E.

**Table of contents**

1. [Import and helpers](#import-and-helpers)
2. [Construction via coherent access to $P$](#coherent-access-p)
3. [Half-step Szegedy quantum walk](#half-step-walk)
4. [Construction via coherent access to $T$](#coherent-access-t)
5. [Efficient coherent access to the Hamiltonian simulation proposal](#hamiltonian-proposal)
6. [Coherent access to the averaged Hamiltonian simulation proposal](#averaged-proposal)


<a id="import-and-helpers"></a>
## Import and helpers

We use NumPy and SciPy for matrix calculations, PennyLane for the quantum circuits, and the existing `monaqa2.mcmc` functions for the Ising model and Markov matrices.


In [2]:
import sys

if ".." not in sys.path:
    sys.path.append("..")

import numpy as np
import pennylane as qml
import scipy.linalg

from monaqa2.mcmc.model import IsingModel
from monaqa2.mcmc.proposal import create_proposal_matrix_local, create_proposal_matrix_quantum_exact
from monaqa2.mcmc.transition import create_acceptance_matrix, create_transition_matrix
from monaqa2.mcmc.utils_hamiltonian import get_cached_mixing_hamiltonian

## Construction via coherent access to $P$

<a id="coherent-access-p"></a>Let $V_P$ be a unitary whose action on states $\ket{x}_a\ket{0^n}_b$ is defined by
$$
V_P\ket{x}_a\ket{0^n}_b
=\ket{x}_a\sum_y\sqrt{P_{yx}}\ket{y}_b.
$$

For testing purposes only, we use a correct but inefficient circuit that generates this coherent access for any column-stochastic matrix $P$. It applies one controlled state-preparation operation for each basis state $x$. The value stored in register $a$ selects $x$, while the operation on register $b$ prepares the amplitudes $\sqrt{P_{yx}}$.


In [3]:
def bits(x, n):
    """Return x as n big-endian bits."""
    return [int(bit) for bit in f"{x:0{n}b}"]
    

def V_P(P, a, b):
    """Prepare |x> sum_y sqrt(P_yx)|y> on explicit registers a and b."""
    n = len(a)
    for x in range(len(P)):
        qml.ctrl(qml.StatePrep, control=a, control_values=bits(x, n))(
            np.sqrt(P[:, x]), wires=b
        )

Let $S$ be the swap operator, defined by $S\ket{x}_a\ket{y}_b=\ket{y}_a\ket{x}_b$.


In [4]:
def S(a, b):
    """Swap two equally sized registers."""
    for aj, bj in zip(a, b):
        qml.SWAP(wires=[aj, bj])

The operator $R_0$ reflects about the all-zero state of register $b$. The circuit also accepts an optional parameter register $q$ and an acceptance coin, which are used in later constructions.


In [5]:
def R0(b, q=None, coin=None):
    """Reflect about zero on b and any explicit q and coin ancillas."""
    ancillas = b + ([] if q is None else q)
    ancillas += [] if coin is None else [coin]
    qml.Reflection(qml.Identity(wires=ancillas), alpha=np.pi)

We can now construct the reflections $R_{\mathcal A}=V_PR_0V_P^\dagger$ and $R_{\mathcal B}=SR_{\mathcal A}S$. 

They reflect about $\mathcal A=V_P(\mathcal H_a\otimes\ket{0^n}_b)$ and $\mathcal B=S\mathcal A$, respectively. 

Let $\mathcal K=\mathcal A+\mathcal B$. The full Szegedy quantum walk, where “full” distinguishes it from the half walk introduced later, is
$$
W_{\mathrm{sz}}=R_{\mathcal B}R_{\mathcal A}.
$$


In [6]:
def R_A(P, a, b):
    """Implement the reflection V_P R_0 V_P^dagger."""
    qml.adjoint(V_P)(P, a, b)
    R0(b)
    V_P(P, a, b)


def R_B(P, a, b):
    """Implement the reflection S R_A S."""
    S(a, b)
    R_A(P, a, b)
    S(a, b)


def full_walk_P(P, a, b):
    """Implement the usual two-reflection Szegedy walk R_B R_A."""
    R_A(P, a, b)
    R_B(P, a, b)

We test this construction by instantiating a small model and constructing its transition matrix $P$.

Any ergodic and reversible transition matrix may be used. Here, $P$ is induced by the local (single spin flip) proposal matrix $T$ and the Metropolis–Hastings acceptance rule at inverse temperature $\beta=1$.


In [23]:
beta = 1.0
model = IsingModel(
    J=np.array([[0.0, 0.8], [0.8, 0.0]]),
    h=np.array([0.35, -0.20]),
)

A = create_acceptance_matrix(model, beta)
T_local = create_proposal_matrix_local(model.n, k=1)
P_local = create_transition_matrix(T_local, model, beta)

Using these data, we construct the matrix associated with $W_{\mathrm{sz}}$. The following method extracts the matrix of a PennyLane circuit in a specified wire order.


In [24]:
def matrix(circuit, *args, wire_order):
    """Return the matrix generated by a PennyLane circuit."""
    return np.asarray(qml.matrix(circuit, wire_order=wire_order)(*args))

We allocate the two $n$-qubit registers $a$ and $b$, then construct $W_{\mathrm{sz}}$ using the wire order $a+b$.


In [25]:
n = int(np.log2(len(P_local)))
a = list(range(n))
b = list(range(n, 2 * n))
wire_order = a + b
W_full = matrix(full_walk_P, P_local, a, b, wire_order=wire_order)

We now verify several properties of this construction.

1. The restriction of $W_{\mathrm{sz}}$ to $\mathcal K^\perp$ is the identity.

   The subspace $\mathcal K^\perp$ is the common $-1$ eigenspace of $R_{\mathcal A}$ and $R_{\mathcal B}$. Thus, its vectors satisfy $(R_{\mathcal A}+\mathbb I)v=0$ and $(R_{\mathcal B}+\mathbb I)v=0$. Since $W_{\mathrm{sz}}=R_{\mathcal B}R_{\mathcal A}$, it follows that $W_{\mathrm{sz}}v=v$.


In [26]:
I = np.eye(len(W_full))
RA = matrix(R_A, P_local, a, b, wire_order=wire_order)
RB = matrix(R_B, P_local, a, b, wire_order=wire_order)
K_perp = scipy.linalg.null_space(np.vstack((RA + I, RB + I)))

if np.allclose(W_full @ K_perp, K_perp):
    print("W_full acts trivially on the subspace K_perp")
else:
    print("Something's wrong...")

W_full acts trivially on the subspace K_perp


2. As an intermediate check, we verify that $V_P^\dagger S V_P$ is a block-encoding of $X$.


In [27]:
def get_block_encoding(U, ancillas):
    system_dimension = len(U) // 2**ancillas
    indices = np.arange(system_dimension) << ancillas
    return U[np.ix_(indices, indices)]

def X(P):
    """Return the symmetric discriminant matrix of P."""
    return np.sqrt(P * P.T)


VP = matrix(V_P, P_local, a, b, wire_order=wire_order)
SM = matrix(S, a, b, wire_order=wire_order)
block = get_block_encoding(VP.conj().T @ SM @ VP, n)

if np.allclose(block, X(P_local)):
    print("V_P^dag @ S @ V_P block-encodes X")
else:
    print("Something's wrong...")

V_P^dag @ S @ V_P block-encodes X


3. Define the isometries $Q_A=V_P(\mathbb I_a\otimes\ket{0^n}_b)$ and $Q_B=SQ_A$. Equivalently, the previous block-encoding identity can be written as $Q_A^\dagger Q_B=X$.


In [28]:
I_A = np.eye(len(P_local))
ket0 = np.eye(len(P_local))[:, [0]]
QA = VP @ np.kron(I_A, ket0)
QB = SM @ QA

if np.allclose(QA.conj().T @ QB, X(P_local)):
    print("Q_A^dag Q_B equals X")
else:
    print("Something's wrong...")

Q_A^dag Q_B equals X


4. Let $\{\ket{v_j}\}_j$ be an orthonormal eigenbasis of $X$, with $X\ket{v_j}=\lambda_j\ket{v_j}$, and define $\ket{a_j}=Q_A\ket{v_j}$ and $\ket{b_j}=Q_B\ket{v_j}$. Then
$$
\braket{a_j}{b_k}
=\bra{v_j}Q_A^\dagger Q_B\ket{v_k}
=\lambda_k\delta_{jk}.
$$


In [29]:
lambdas, vs = np.linalg.eigh(X(P_local))
a_states = QA @ vs
b_states = QB @ vs

if np.allclose(a_states.conj().T @ b_states, np.diag(lambdas)):
    print("The overlaps <a_j|b_k> equal lambda_k delta_jk")
else:
    print("Something's wrong...")

The overlaps <a_j|b_k> equal lambda_k delta_jk


5. Let $\lambda_j=\cos(\theta_j)$ and define $\mathcal K_j=\operatorname{span}\{\ket{a_j},\ket{b_j}\}$.

    For $|\lambda_j|<1$, define
    $$\ket{a_j^\perp}=(\ket{b_j}-\cos(\theta_j)\ket{a_j})/\sin(\theta_j).$$

    Then $\{\ket{a_j},|a_j^\perp\rangle\}$ is an orthonormal basis of $\mathcal K_j$, and in this basis
$$
W_{\mathrm{sz}}|_{\mathcal K_j}
=
\begin{pmatrix}
\cos(2\theta_j) & -\sin(2\theta_j) \\
\sin(2\theta_j) & \cos(2\theta_j)
\end{pmatrix}.
$$
Therefore, its eigenvalues are $e^{\pm 2i\theta_j}$.

    For $|\lambda_j|=1$, then $\ket{b_j}=\lambda_j\ket{a_j}$, so $\mathcal K_j$ is one-dimensional and $W_{\mathrm{sz}}|_{\mathcal K_j}=\mathbb I_{\mathcal K_j}$.

In [30]:
ok = True

for j, lambda_j in enumerate(lambdas):
    a_j = a_states[:, j]
    b_j = b_states[:, j]
    theta_j = np.arccos(np.clip(lambda_j, -1, 1))

    if np.isclose(abs(lambda_j), 1):
        ok &= np.allclose(b_j, lambda_j * a_j)
        ok &= np.allclose(W_full @ a_j, a_j)
    else:
        a_perp_j = (b_j - np.cos(theta_j) * a_j) / np.sin(theta_j)
        basis = np.column_stack((a_j, a_perp_j))
        rotation = np.array([
            [np.cos(2 * theta_j), -np.sin(2 * theta_j)],
            [np.sin(2 * theta_j),  np.cos(2 * theta_j)],
        ])
        ok &= np.allclose(W_full @ basis, basis @ rotation)

if ok:
    print("The restriction of W_full to every K_j has the expected form")
else:
    print("Something's wrong...")

The restriction of W_full to every K_j has the expected form


## Half-step Szegedy quantum walk

<a id="half-step-walk"></a>Using $R_{\mathcal B}=SR_{\mathcal A}S$, the full walk is $W_{\mathrm{sz}}=SR_{\mathcal A}SR_{\mathcal A}$. We define the half-step walk as
$$
W_P=SR_{\mathcal A}=SV_PR_0V_P^\dagger,
\qquad
W_P^2=W_{\mathrm{sz}}.
$$


The circuit applies $R_{\mathcal A}$ followed by the register swap $S$.


In [31]:
def half_walk_P(P, a, b):
    """Implement the half-step walk S R_A."""
    R_A(P, a, b)
    S(a, b)

We first construct $W_P$ and verify directly that its square is the full walk.


In [32]:
W_half = matrix(half_walk_P, P_local, a, b, wire_order=wire_order)

if np.allclose(W_half @ W_half, W_full):
    print("W_half squared equals W_full")
else:
    print("Something's wrong...")

W_half squared equals W_full


Since $S\ket{a_j}=\ket{b_j}$, the half walk preserves each $\mathcal K_j$. For $|\lambda_j|<1$, in the orthonormal basis $\{\ket{a_j},\ket{a_j^\perp}\}$,
$$
W_P|_{\mathcal K_j}
=
\begin{pmatrix}
\cos(\theta_j) & -\sin(\theta_j) \\
\sin(\theta_j) & \cos(\theta_j)
\end{pmatrix},
$$
so its eigenvalues are $e^{\pm i\theta_j}=e^{\pm i\arccos(\lambda_j)}$. For $|\lambda_j|=1$, the one-dimensional block has eigenvalue $\lambda_j$.


In [33]:
ok = True

for lambda_j, a_j, b_j in zip(lambdas, a_states.T, b_states.T):
    theta_j = np.arccos(np.clip(lambda_j, -1, 1))
    if np.isclose(abs(lambda_j), 1):
        ok &= np.allclose(W_half @ a_j, lambda_j * a_j)
    else:
        basis = np.c_[a_j, (b_j - lambda_j * a_j) / np.sin(theta_j)]
        rotation = [[np.cos(theta_j), -np.sin(theta_j)],
                    [np.sin(theta_j),  np.cos(theta_j)]]
        ok &= np.allclose(W_half @ basis, basis @ rotation)

print("Restriction of W_half to every K_j:", "OK" if ok else "FAILED")

Restriction of W_half to every K_j: OK


## Construction via coherent access to $T$

<a id="coherent-access-t"></a>Direct coherent access to $P$ is inconvenient because its diagonal contains the sum of all rejected proposals. Instead, we coherently access only the proposal matrix $T$ through
$$
V\ket{x}_a\ket{0^n}_b
=
\ket{x}_a\sum_y\sqrt{T_{yx}}\ket{y}_b.
$$
The same circuit $V_P$ that we have implemented earlier can be reused for this operation, simply call the method passing the matrix $T$ instead of the matrix $P$. 

This time, the acceptance probability enter the picture is introduced through a coin qubit (in contrast to the previous case that had the acceptance probabilities already embedded in the coherent access $V_P$):
$$
B\ket{x}_a\ket{y}_b\ket{0}_c
=
\ket{x}_a\ket{y}_b
\left(
\sqrt{1-A_{yx}}\ket{0}_c
+
\sqrt{A_{yx}}\ket{1}_c
\right).
$$


In [34]:
def B(A, a, b, coin):
    """Prepare the coherent acceptance amplitude."""
    n = len(a)
    for x in range(len(A)):
        for y in range(len(A)):
            if A[y, x] == 0:
                continue
            qml.ctrl(
                qml.RY,
                control=a + b,
                control_values=bits(x, n) + bits(y, n),
            )(2 * np.arcsin(np.sqrt(A[y, x])), wires=coin)

The unitary $F$ applies the accepted move by swapping registers $a$ and $b$ when the coin is $\ket{1}_c$.


In [35]:
def F(a, b, coin):
    """Swap a and b when the acceptance coin is one."""
    for aj, bj in zip(a, b):
        qml.CSWAP(wires=[coin, aj, bj])

The resulting walk is $W_T=R_0V^\dagger B^\dagger FBV$, where $R_0$ reflects about the joint zero state of registers $b$ and $c$.


In [36]:
def walk_T(T, A, a, b, coin):
    """Implement R_0 V^dagger B^dagger F B V."""
    V_P(T, a, b)
    B(A, a, b, coin)
    F(a, b, coin)
    qml.adjoint(B)(A, a, b, coin)
    qml.adjoint(V_P)(T, a, b)
    R0(b, coin=coin)

We construct $W_T$ on registers $a$, $b$, and $c$.


In [37]:
n = int(np.log2(len(T_local)))
a = list(range(n))
b = list(range(n, 2 * n))
coin = 2 * n
wire_order = a + b + [coin]

W_T = matrix(walk_T, T_local, A, a, b, coin, wire_order=wire_order)

The zero-ancilla block of $W_T$ can be evaluated directly:
* For $x\neq y$, only accepted proposals contribute, giving
$$\sqrt{T_{xy}A_{xy}}\sqrt{T_{yx}A_{yx}}=\sqrt{P_{xy}P_{yx}}=X_{xy}.$$
* For $x=y$, the self-proposal and all rejected proposals contribute
$$T_{xx}+\sum_{z\neq x}T_{zx}(1-A_{zx})=P_{xx}=X_{xx}.$$

Therefore,
$$
(\mathbb I_a\otimes\bra{0}_{bc})W_T
(\mathbb I_a\otimes\ket{0}_{bc})
=X.
$$

In [38]:
I_A = np.eye(len(P_local))
ket0_bc = np.eye(2 ** (n + 1))[:, [0]]
iso = np.kron(I_A, ket0_bc)

if np.allclose(iso.conj().T @ W_T @ iso, X(P_local)):
    print("W_T block-encodes X")
else:
    print("Something's wrong...")

W_T block-encodes X


Since $F^2=\mathbb I$, the operator $R_0W_T=V^\dagger B^\dagger FBV$ is an involution.

In [39]:
zero_bc = np.kron(I_A, ket0_bc)
R0_T = 2 * zero_bc @ zero_bc.conj().T - np.eye(len(W_T))

if np.allclose(R0_T @ W_T @ R0_T @ W_T, np.eye(len(W_T))):
    print("R_0 W_T is an involution")
else:
    print("Something's wrong...")

R_0 W_T is an involution


Let $X\ket{v_j}=\lambda_j\ket{v_j}$, define $\ket{\phi_j}=\ket{v_j}_a\ket{0}_{bc}$, and write $\lambda_j=\cos(\theta_j)$.

**For $|\lambda_j|<1$**: the zero-ancilla component of $R_0W_T\ket{\phi_j}$ is $\lambda_j\ket{\phi_j}$. Removing it leaves a component of norm $\sin(\theta_j)$. We therefore define $\ket{\phi_j^\perp}=(R_0W_T\ket{\phi_j}-\lambda_j\ket{\phi_j})/\sin(\theta_j)$. The states $\ket{\phi_j}$ and $\ket{\phi_j^\perp}$ span an invariant two-dimensional subspace $\mathcal K_j$. In this ordered basis,
$$
W_T|_{\mathcal K_j}
=
\begin{pmatrix}
\cos(\theta_j) & \sin(\theta_j) \\
-\sin(\theta_j) & \cos(\theta_j)
\end{pmatrix},
$$
whose eigenvalues are $e^{\pm i\theta_j}$.

**For $|\lambda_j|=1$**: we have $\sin(\theta_j)=0$, so there is no second basis state. Instead, $R_0W_T\ket{\phi_j}=\lambda_j\ket{\phi_j}$ and therefore $W_T\ket{\phi_j}=\lambda_j\ket{\phi_j}$. Thus, $\mathcal K_j=\operatorname{span}\{\ket{\phi_j}\}$ is one-dimensional and $W_T|_{\mathcal K_j}=\lambda_j\mathbb I_{\mathcal K_j}$.

In [41]:
j = 0

while j < len(lambdas):
    lambda_j = lambdas[j]
    theta_j = np.arccos(np.clip(lambda_j, -1, 1))
    phi_j = zero_bc @ vs[:, j]

    # Case 1: two-dimensional invariant subspace
    if not np.isclose(abs(lambda_j), 1):
        phi_perp_j = (
            R0_T @ W_T @ phi_j - lambda_j * phi_j
        ) / np.sin(theta_j)

        basis = np.column_stack([phi_j, phi_perp_j])
        restriction = basis.conj().T @ W_T @ basis
        expected = np.array([
            [np.cos(theta_j),  np.sin(theta_j)],
            [-np.sin(theta_j), np.cos(theta_j)],
        ])

        matches = np.allclose(restriction, expected)
        outcome = "OK, rotation matches" if matches else "FAILED"

    # Case 2: one-dimensional invariant subspace
    else:
        matches = np.allclose(W_T @ phi_j, lambda_j * phi_j)
        outcome = "OK, endpoint eigenvalue matches" if matches else "FAILED"

    print(f"K_{j}: lambda={lambda_j: .6f}, theta={theta_j:.6f} | {outcome}")
    j += 1

K_0: lambda=-0.000000, theta=1.570796 | OK, rotation matches
K_1: lambda= 0.098157, theta=1.472481 | OK, rotation matches
K_2: lambda= 0.901843, theta=0.446780 | OK, rotation matches
K_3: lambda= 1.000000, theta=0.000000 | OK, endpoint eigenvalue matches


## Efficient coherent access to the Hamiltonian simulation proposal

<a id="hamiltonian-proposal"></a>For $U(t,\gamma)=e^{-itH_{\mathrm{tf}}(\gamma)}$, the proposal matrix is $T_{yx}=|\bra{y}U(t,\gamma)\ket{x}|^2$. 

We define our coherent access as the unitary $V_U$,
$$
V_U\ket{x}_a\ket{0^n}_b
=
\ket{x}_aU\ket{x}_b
=
\ket{x}_a\sum_y U_{yx}\ket{y}_b.
$$


In [46]:
def V_U(U, a, b):
    """Prepare |x> U|x> on registers a and b."""
    for aj, bj in zip(a, b):
        qml.CNOT(wires=[aj, bj])
    qml.QubitUnitary(U, wires=b)

We construct one Hamiltonian simulation unitary $U$, its proposal matrix $T$, and the corresponding Metropolis transition matrix $P$.

In [47]:
H_ising = np.diag(model.energies_rescaled)
H_mixer = get_cached_mixing_hamiltonian(model.n)
gamma, t = 0.57, 0.83

U = scipy.linalg.expm(-1j * t * ((1 - gamma) * H_ising + gamma * H_mixer))
T = create_proposal_matrix_quantum_exact(
    model.h_rescaled, model.J_rescaled, gamma=gamma, t=t
)
P = create_transition_matrix(T, model, beta)

The Hamiltonian is symmetric in the computational basis, so $U^\top=U$. We also verify directly that the classical proposal returned by monaqa2 is $T=|U|^2$.


In [48]:
if np.allclose(U, U.T):
    print("U is symmetric")
else:
    print("Something's wrong...")

if np.allclose(T, np.abs(U) ** 2):
    print("T equals |U|^2")
else:
    print("Something's wrong...")

U is symmetric
T equals |U|^2


Replacing the usual preparation $V$ by $V_U$ gives $W_U=R_0V_U^\dagger B^\dagger FBV_U$.

In [49]:
def walk_U(U, A, a, b, coin):
    """Implement R_0 V_U^dagger B^dagger F B V_U."""
    V_U(U, a, b)
    B(A, a, b, coin)
    F(a, b, coin)
    qml.adjoint(B)(A, a, b, coin)
    qml.adjoint(V_U)(U, a, b)
    R0(b, coin=coin)

Because $U^\top=U$, the phases of $U_{xy}$ and $U_{yx}$ cancel in the off-diagonal compressed matrix elements. The diagonal terms are unchanged, so the zero-ancilla block of $W_U$ is again $X$.


In [50]:
n = int(np.log2(len(T)))
a = list(range(n))
b = list(range(n, 2 * n))
coin = 2 * n
wire_order = a + b + [coin]

W_U = matrix(walk_U, U, A, a, b, coin, wire_order=wire_order)

if np.allclose(get_block_encoding(W_U, n + 1), X(P)):
    print("W_U block-encodes X")
else:
    print("Something's wrong...")

W_U block-encodes X


For each eigenvalue $\lambda_j=\cos(\theta_j)$ of $X$, we verify that the corresponding block of $W_U$ has phases $\pm\theta_j$. For $|\lambda_j|=1$, the block is one-dimensional with phase $0$ or $\pi$.

In [53]:
lambdas_U, vectors_U = np.linalg.eigh(X(P))

I_a = np.eye(len(P))
ket0_bc = np.eye(2 ** (n + 1))[:, [0]]
zero_bc = np.kron(I_a, ket0_bc)
R0_U = 2 * zero_bc @ zero_bc.conj().T - np.eye(len(W_U))

j = 0

while j < len(lambdas_U):
    lambda_j = lambdas_U[j]
    theta_j = np.arccos(np.clip(lambda_j, -1, 1))
    phi_j = zero_bc @ vectors_U[:, j]

    # Two-dimensional block: phases are -theta_j and +theta_j.
    if not np.isclose(abs(lambda_j), 1, atol=1e-10, rtol=0):
        phi_perp_j = (R0_U @ W_U @ phi_j - lambda_j * phi_j) / np.sin(theta_j)
        basis = np.column_stack([phi_j, phi_perp_j])
        restriction = basis.conj().T @ W_U @ basis
        phases = np.sort(np.angle(np.linalg.eigvals(restriction)))
        expected_phases = np.array([-theta_j, theta_j])

    # One-dimensional block: the phase is zero or pi.
    else:
        eigenvalue = np.vdot(phi_j, W_U @ phi_j)
        phases = np.array([abs(np.angle(eigenvalue))])
        expected_phases = np.array([0.0 if lambda_j > 0 else np.pi])

    matches = np.allclose(phases, expected_phases, atol=1e-7)
    outcome = "OK, Szegedy phases match" if matches else "FAILED"
    print(f"K_{j}: lambda={lambda_j:.6f}, phases={phases} | {outcome}")

    j += 1

K_0: lambda=0.681530, phases=[-0.82094429  0.82094429] | OK, Szegedy phases match
K_1: lambda=0.700043, phases=[-0.79533845  0.79533845] | OK, Szegedy phases match
K_2: lambda=0.928896, phases=[-0.37937626  0.37937626] | OK, Szegedy phases match
K_3: lambda=1.000000, phases=[2.09408262e-19] | OK, Szegedy phases match


## Coherent access to the averaged Hamiltonian-simulation proposal

<a id="averaged-proposal"></a>The randomized choice of the hyperparameters $(t,\gamma)$ is implemented by coherently selecting among unitaries associated with a discretized grid of parameter values.

Here we test a small-scale construction using four unitaries, corresponding to four possible parameter pairs. For four symmetric proposal unitaries, the averaged proposal matrix is
$$
\bar T_{yx}
=
\frac{1}{4}\sum_{\mu=0}^{3}
\left|\bra{y}U_\mu\ket{x}\right|^2.
$$

We start by explicitly constructing the four unitaries, their proposal matrices, $\bar T$, and the associated transition matrix $\bar P$.

In [57]:
gamma_0, t_0 = 0.25, 0.40
gamma_1, t_1 = 0.45, 0.70
gamma_2, t_2 = 0.65, 1.00
gamma_3, t_3 = 0.80, 1.30

U_0 = scipy.linalg.expm(-1j * t_0 * ((1 - gamma_0) * H_ising + gamma_0 * H_mixer))
U_1 = scipy.linalg.expm(-1j * t_1 * ((1 - gamma_1) * H_ising + gamma_1 * H_mixer))
U_2 = scipy.linalg.expm(-1j * t_2 * ((1 - gamma_2) * H_ising + gamma_2 * H_mixer))
U_3 = scipy.linalg.expm(-1j * t_3 * ((1 - gamma_3) * H_ising + gamma_3 * H_mixer))

T_0 = create_proposal_matrix_quantum_exact(
    model.h_rescaled, model.J_rescaled, gamma=gamma_0, t=t_0
)
T_1 = create_proposal_matrix_quantum_exact(
    model.h_rescaled, model.J_rescaled, gamma=gamma_1, t=t_1
)
T_2 = create_proposal_matrix_quantum_exact(
    model.h_rescaled, model.J_rescaled, gamma=gamma_2, t=t_2
)
T_3 = create_proposal_matrix_quantum_exact(
    model.h_rescaled, model.J_rescaled, gamma=gamma_3, t=t_3
)

U_list = [U_0, U_1, U_2, U_3]
T_bar = (T_0 + T_1 + T_2 + T_3) / 4
P_bar = create_transition_matrix(T_bar, model, beta)

With 4 unitaries, we need an additional two-qubit register $q$ that prepares the uniform superposition over the four parameter choices and controls the corresponding unitary:
$$
\bar V\ket{x}_a\ket{0^n}_b\ket{0^2}_q
=
\frac{1}{2}\sum_{\mu=0}^{3}
\ket{x}_aU_\mu\ket{x}_b\ket{\mu}_q.
$$

Here the construction is simplified: the prepare unitary is simply given by two hadamard on the auxiliary register $q$. 

In [58]:
def V_bar(U_list, a, b, q):
    """Prepare the coherent average of four proposal unitaries."""
    U_0, U_1, U_2, U_3 = U_list

    for aj, bj in zip(a, b):
        qml.CNOT(wires=[aj, bj])

    qml.Hadamard(wires=q[0])
    qml.Hadamard(wires=q[1])
    qml.ctrl(qml.QubitUnitary, control=q, control_values=bits(0, 2))(U_0, wires=b)
    qml.ctrl(qml.QubitUnitary, control=q, control_values=bits(1, 2))(U_1, wires=b)
    qml.ctrl(qml.QubitUnitary, control=q, control_values=bits(2, 2))(U_2, wires=b)
    qml.ctrl(qml.QubitUnitary, control=q, control_values=bits(3, 2))(U_3, wires=b)

The averaged walk is $\bar W=R_0\bar V^\dagger B^\dagger FB\bar V$, where $R_0$ reflects about the joint zero state of registers $b$, $q$, and $c$.


In [59]:
def walk_bar(U_list, A, a, b, q, coin):
    """Implement R_0 V_bar^dagger B^dagger F B V_bar."""
    V_bar(U_list, a, b, q)
    B(A, a, b, coin)
    F(a, b, coin)
    qml.adjoint(B)(A, a, b, coin)
    qml.adjoint(V_bar)(U_list, a, b, q)
    R0(b, q=q, coin=coin)


We first verify that all four unitaries are symmetric and that $\bar T$ is the average of their elementwise squared moduli.


In [60]:
symmetric = (
    np.allclose(U_0, U_0.T)
    and np.allclose(U_1, U_1.T)
    and np.allclose(U_2, U_2.T)
    and np.allclose(U_3, U_3.T)
)
expected_T_bar = (
    np.abs(U_0) ** 2
    + np.abs(U_1) ** 2
    + np.abs(U_2) ** 2
    + np.abs(U_3) ** 2
) / 4

print("All four U_mu are symmetric:", "OK" if symmetric else "FAILED")
print("T_bar is their averaged proposal:", "OK" if np.allclose(T_bar, expected_T_bar) else "FAILED")

All four U_mu are symmetric: OK
T_bar is their averaged proposal: OK


Orthogonality of the states $\ket{\mu}_q$ removes cross terms between different unitaries. Since each $U_\mu$ is symmetric, the remaining phases cancel term by term, and the zero-ancilla block of $\bar W$ is the discriminant $\bar X$ of $\bar P$.


In [64]:
n = int(np.log2(len(T_bar)))
k = 2
a = list(range(n))
b = list(range(n, 2 * n))
q = list(range(2 * n, 2 * n + k))
coin = 2 * n + k
wire_order = a + b + q + [coin]

W_bar = matrix(walk_bar, U_list, A, a, b, q, coin, wire_order=wire_order)

if np.allclose(get_block_encoding(W_bar, n + k + 1), X(P_bar)):
    print("W_bar block-encodes X_bar")
else:
    print("Something's wrong...")

W_bar block-encodes X_bar


For each eigenvalue $\bar\lambda_j=\cos(\bar\theta_j)$ of $\bar X$, we verify that the corresponding block of $W_{\mathrm{bar}}$ has Szegedy phases $\pm\bar\theta_j$, with phase $0$ or $\pi$ at the endpoints.

In [65]:
lambdas_bar, vectors_bar = np.linalg.eigh(X(P_bar))

I_a = np.eye(len(P_bar))
ket0_bqc = np.eye(2 ** (n + k + 1))[:, [0]]
zero_bqc = np.kron(I_a, ket0_bqc)
R0_bar = 2 * zero_bqc @ zero_bqc.conj().T - np.eye(len(W_bar))

j = 0

while j < len(lambdas_bar):
    lambda_j = lambdas_bar[j]
    theta_j = np.arccos(np.clip(lambda_j, -1, 1))
    phi_j = zero_bqc @ vectors_bar[:, j]

    # Two-dimensional block: phases are -theta_j and +theta_j.
    if not np.isclose(abs(lambda_j), 1, atol=1e-10, rtol=0):
        phi_perp_j = (R0_bar @ W_bar @ phi_j - lambda_j * phi_j) / np.sin(theta_j)
        basis = np.column_stack([phi_j, phi_perp_j])
        restriction = basis.conj().T @ W_bar @ basis
        phases = np.sort(np.angle(np.linalg.eigvals(restriction)))
        expected_phases = np.array([-theta_j, theta_j])

    # One-dimensional block: the phase is zero or pi.
    else:
        eigenvalue = np.vdot(phi_j, W_bar @ phi_j)
        phases = np.array([abs(np.angle(eigenvalue))])
        expected_phases = np.array([0.0 if lambda_j > 0 else np.pi])

    matches = np.allclose(phases, expected_phases, atol=1e-7)
    outcome = "OK, Szegedy phases match" if matches else "FAILED"
    print(f"K_{j}: lambda={lambda_j:.6f}, phases={phases} | {outcome}")

    j += 1

K_0: lambda=0.611536, phases=[-0.9127964  0.9127964] | OK, Szegedy phases match
K_1: lambda=0.756129, phases=[-0.71341942  0.71341942] | OK, Szegedy phases match
K_2: lambda=0.811985, phases=[-0.62325217  0.62325217] | OK, Szegedy phases match
K_3: lambda=1.000000, phases=[5.30455966e-20] | OK, Szegedy phases match
